# XAI-Verify — Train BERT & RoBERTa on Google Colab

**Before you start:** Runtime → **Change runtime type** → **T4 GPU** → Save

**Upload from your PC** (one of these):
- **Best:** zip the whole folder `deepfake_detection_project` and upload in cell 2, **or**
- Upload `scripts.zip` + `WELFake_Dataset.csv` + zip of `utils/` folder in cell 3

After training, download `fake_news_models.zip` and extract into `trained_models/` on your PC.

In [ ]:
# Cell 1 — GPU check (if cuda False, see markdown above)
import shutil, torch
if shutil.which("nvidia-smi"):
    !nvidia-smi
else:
    print("nvidia-smi not found — this runtime has NO GPU.")
    print("Colab: Runtime → Change runtime type → T4 GPU → Save, then Runtime → Restart session.")
USE_GPU = torch.cuda.is_available()
print("cuda:", USE_GPU)
if not USE_GPU:
    print("\nYou can still train on CPU with a small subset (cell 6 uses --max-train 4000).")

In [ ]:
# Cell 2 — Upload FULL project zip (recommended)
from google.colab import files
import glob, os, zipfile

print("Select deepfake_detection_project.zip (entire project folder)")
uploaded = files.upload()

zips = [f for f in uploaded if f.lower().endswith(".zip")]
assert zips, "Please upload a .zip file"

for z in zips:
    zipfile.ZipFile(z).extractall("/content")

matches = glob.glob("/content/**/train_fake_news_transformers.py", recursive=True)
assert matches, "Zip must contain scripts/train_fake_news_transformers.py"

PROJECT = matches[0].split("/scripts/")[0]
os.chdir(PROJECT)
print("Project root:", PROJECT)

In [ ]:
# Cell 3 — OR upload missing pieces (skip if cell 2 used full project zip)
from google.colab import files
import os, shutil, zipfile, glob

PROJECT = globals().get("PROJECT", "/content")
os.makedirs(f"{PROJECT}/datasets/text", exist_ok=True)
os.makedirs(f"{PROJECT}/scripts", exist_ok=True)

print("Upload scripts.zip (if needed)")
up1 = files.upload()
for name in up1:
    if name.lower().endswith(".zip"):
        zipfile.ZipFile(name).extractall(PROJECT)

print("Upload WELFake_Dataset.csv")
up2 = files.upload()
for name in up2:
    if name.lower().endswith(".csv"):
        dest = f"{PROJECT}/datasets/text/WELFake_Dataset.csv"
        shutil.move(name, dest)
        print("Saved", dest)

print("Upload utils.zip (folder utils/ with text_ml.py inside)")
up3 = files.upload()
for name in up3:
    if name.lower().endswith(".zip"):
        zipfile.ZipFile(name).extractall(PROJECT)

matches = glob.glob(f"{PROJECT}/**/train_fake_news_transformers.py", recursive=True)
if matches:
    PROJECT = matches[0].split("/scripts/")[0]
os.chdir(PROJECT)
print("Project root:", PROJECT)

In [ ]:
# Cell 4 — Preflight (must pass before training)
import os
from pathlib import Path

PROJECT = Path(os.getcwd())
checks = {
    "scripts/train_fake_news_transformers.py": PROJECT / "scripts/train_fake_news_transformers.py",
    "scripts/train_text_benchmark.py": PROJECT / "scripts/train_text_benchmark.py",
    "utils/text_ml.py": PROJECT / "utils/text_ml.py",
    "datasets/text/WELFake_Dataset.csv": PROJECT / "datasets/text/WELFake_Dataset.csv",
}
ok = True
for label, path in checks.items():
    status = "OK" if path.is_file() else "MISSING"
    print(f"{status:7} {label}")
    if status == "MISSING":
        ok = False
assert ok, "Fix missing files (run cell 2 or 3), then re-run this cell."

In [ ]:
# Cell 5 — Install dependencies
!pip install -q torch transformers scikit-learn numpy accelerate

In [ ]:
# Cell 6 — Train BERT + RoBERTa
import torch
if torch.cuda.is_available():
    !python scripts/train_fake_news_transformers.py --architecture both --epochs 3 --batch-size 16
else:
    print("CPU mode: smaller data, 1 epoch each — enable T4 GPU for full training.")
    !python scripts/train_fake_news_transformers.py --architecture both --epochs 1 --batch-size 8 --max-train 4000

In [ ]:
# Cell 7 — Verify weights saved
from pathlib import Path
for name in ["fake_news_bert_welfake", "fake_news_roberta_welfake"]:
    d = Path("trained_models") / name
    weights = list(d.glob("model.safetensors")) + list(d.glob("pytorch_model.bin"))
    print(name, "OK" if weights else "MISSING WEIGHTS", "→", d)

In [ ]:
# Cell 8 — Download zip to your PC
from google.colab import files
import shutil
from pathlib import Path

out = "/content/fake_news_models.zip"
shutil.make_archive("/content/fake_news_models", "zip", "trained_models")
print("Download starting...")
files.download(out)
print("On PC: unzip into F:\\deepfake_detection_project\\trained_models\\")